# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [24]:
import warnings
warnings.filterwarnings('ignore')

In [25]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [26]:
#!pip install pandas

In [27]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [28]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## LLMChain

In [29]:
!pip install langchain_community

In [30]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [31]:
import httpx
ipv4_client = httpx.Client(transport=httpx.HTTPTransport(local_address="0.0.0.0"))

# Justification: Only the default (1) value is supported on this model, so we set it to 1. But we can increase the temperature to make the output more creative and varied.
# but not so much that the text becomes inconsistent or off-topic.
# Adding a custom http client to ensure that the requests are made over IPv4 since IPv6 wasn't working for me.
llm = ChatOpenAI(model="gpt-6-luna", temperature=1, http_client=ipv4_client)

In [32]:
prompt = ChatPromptTemplate.from_template(
    "Write a short, appealing description (2-3 sentences) for the following product: {product}"
)

In [33]:

chain = LLMChain(llm=llm, prompt=prompt)

In [34]:
product = "a reusable water bottle"
chain.run(product)

'Stay hydrated wherever the day takes you with this reusable water bottle. Lightweight, durable, and easy to carry, it’s a simple way to enjoy your favorite drinks while cutting down on single-use plastics.'

## SimpleSequentialChain

In [35]:
from langchain.chains import SimpleSequentialChain

In [ ]:
llm = ChatOpenAI(model="gpt-6-luna", temperature=1, http_client=ipv4_client)

first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}? Only return the name."
)
chain_one = LLMChain(llm=llm, prompt=first_prompt)

second_prompt = ChatPromptTemplate.from_template(
    "Write a one-line slogan for the following company: {company_name}"
)
chain_two = LLMChain(llm=llm, prompt=second_prompt)



In [37]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two], verbose=True)

In [38]:
overall_simple_chain.run(product)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


EverSip

EverSip — Make every moment worth sipping.

> Finished chain.


'EverSip — Make every moment worth sipping.'

In [39]:
overall_simple_chain.run("a standing desk")
overall_simple_chain.run("noise-cancelling headphones")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Standing desk manufacturer


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Rise to your best work.

> Finished chain.
HushTone
HushTone: Find your quiet frequency.

> Finished chain.


'HushTone: Find your quiet frequency.'

**Repeat the above twice for different products**

In [57]:
llm = ChatOpenAI(model="gpt-6-luna", temperature=1, http_client=ipv4_client)

first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}? Only return the name."
)
chain_one = LLMChain(llm=llm, prompt=first_prompt)

second_prompt = ChatPromptTemplate.from_template(
    "Write a one-line slogan for the following company: {company_name}"
)
chain_two = LLMChain(llm=llm, prompt=second_prompt)



In [58]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two], verbose=True)

In [60]:
overall_simple_chain.run("a videogames console")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Video game console manufacturer
Powering play. Expanding worlds.

> Finished chain.


'Powering play. Expanding worlds.'

In [61]:
overall_simple_chain.run("a mechanical keyboard")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Switchworks

Switchworks: Powering smarter connections.

> Finished chain.


'Switchworks: Powering smarter connections.'

## SequentialChain

In [40]:
from langchain.chains import SequentialChain

In [41]:
llm = ChatOpenAI(model="gpt-6-luna", temperature=1, http_client=ipv4_client)

first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to English:\n\n{Review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt,
                     output_key="English_Review"
                    )


In [42]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize the following review in 1 sentence:\n\n{English_Review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt,
                     output_key="summary"
                    )


In [43]:
# prompt template 3: detect the language of the review
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review written in? Only return the language name:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )


In [44]:
# prompt template 4: follow up message that takes as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [45]:
# overall_chain: input= Review
# and output= English_Review, summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary", "followup_message"],
    verbose=True
)


In [46]:
review = df.Review[5]
overall_chain(review)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': 'I find the taste mediocre. The foam doesn’t last, which is strange. I buy the same ones in stores, and they taste much better… An old batch or a counterfeit?!',
 'summary': 'The product tasted mediocre and had unusually short-lived foam compared with store-bought versions, leading the reviewer to suspect it was old or counterfeit.',
 'followup_message': 'Nous sommes désolés que le goût et la tenue de la mousse n’aient pas été à la hauteur de vos attentes. Nous comprenons que cela puisse vous faire douter de la fraîcheur ou de l’authenticité du produit. Pour nous aider à vérifier, pourriez-vous nous communiquer votre numéro de commande ainsi que le numéro de lot et la date de péremption indiqués sur l’emballage ? Nous examinerons la situation et reviendrons vers vous avec une solution.'}

**Repeat the above twice for different products or reviews**

In [62]:
   overall_chain(df.Review[0])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'Review': 'I ordered a king size set. My only criticism would be that I wish seller would offer the king size set with 4 pillowcases. I separately ordered a two pack of pillowcases so I could have a total of four. When I saw the two packages, it looked like the color did not exactly match. Customer service was excellent about sending me two more pillowcases so I would have four that matched. Excellent! For the cost of these sheets, I am satisfied with the characteristics and coolness of the sheets.',
 'English_Review': 'I ordered a king-size set. My only criticism is that I wish the seller offered it with four pillowcases. I ordered a separate two-pack so I’d have four in total. When the packages arrived, the colors didn’t seem to match exactly. Customer service was excellent and sent me two more pillowcases so I’d have four that matched. Excellent service! For the price, I’m happy with the sheets’ quality and how cool they feel.',
 'summary': 'The affordable sheets feel cool and are 

In [63]:
   overall_chain(df.Review[3])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'Review': 'This is the best throw pillow fillers on Amazon. I’ve tried several others, and they’re all cheap and flat no matter how much fluffing you do. Once you toss these in the dryer after you remove them from the vacuum sealed shipping material, they fluff up great',
 'English_Review': 'These are the best throw pillow inserts on Amazon. I’ve tried several others, but they all feel cheap and stay flat no matter how much you fluff them. After removing these from the vacuum-sealed packaging, just toss them in the dryer and they fluff up beautifully.',
 'summary': 'The reviewer says these pillow inserts outperform cheaper, flat alternatives and fluff up beautifully after a quick cycle in the dryer.',
 'followup_message': 'Thank you for the kind review! We’re delighted the inserts are a step up from flatter alternatives and fluffed up so nicely after a quick spin in the dryer. Hope you enjoy them!'}

## Router Chain

In [47]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [48]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [64]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [69]:
llm = ChatOpenAI(model="gpt-6-luna", temperature=1, http_client=ipv4_client)

In [70]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [71]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [72]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [73]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [74]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [75]:
chain.run("What is black body radiation?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


physics: {'input': 'What is black body radiation?'}
> Finished chain.


'Blackbody radiation is the electromagnetic radiation emitted by an ideal object that absorbs all radiation falling on it. Its spectrum depends only on the object’s temperature—not on its material.\n\nAs the object gets hotter, it emits more radiation overall, and the strongest part of its glow shifts toward shorter wavelengths (from infrared toward visible light, for example). A warm stove and a star are approximate examples of objects that emit thermal, blackbody-like radiation.'

In [76]:
chain.run("what is 2 + 2")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


math: {'input': 'what is 2 + 2'}
> Finished chain.


'2 + 2 = 4.'

In [77]:
chain.run("Why does every cell in our body contain DNA?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Most cells in your body contain DNA because they developed from the fertilized egg, whose DNA was copied as cells divided. That DNA provides instructions for building and maintaining the cell and, in many cells, for making the proteins it needs.\n\nDifferent cell types—such as muscle cells and neurons—usually contain nearly the same DNA, but they use different parts of it. This selective gene activity helps give each cell its specialized role.\n\nThere are exceptions: mature red blood cells, for example, lose their nucleus and nuclear DNA as they develop. Most cells also contain a small amount of separate DNA in their mitochondria.'

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [78]:
llm = ChatOpenAI(model="gpt-6-luna", temperature=1, http_client=ipv4_client)

# cell 48 (new cells below it): own questions
chain.run("What is the time complexity of binary search?")
chain.run("Who was Cleopatra and why was she important?")
chain.run("Suggest a recipe for a quick tomato soup.")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


computer science: {'input': 'What is the time complexity of binary search?'}

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.
History: {'input': 'Who was Cleopatra and why was she important?'}

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.
None: {'input': 'Suggest a recipe for a quick tomato soup.'}
> Finished chain.


'## Quick Tomato Soup\n\n**Ready in:** 20 minutes  \n**Serves:** 4\n\n### Ingredients\n- 1 tablespoon olive oil or butter\n- 1 small onion, chopped\n- 2 garlic cloves, minced\n- 1 can (28 oz / 800 g) crushed tomatoes\n- 2 cups vegetable or chicken broth\n- ½ teaspoon sugar, optional\n- Salt and black pepper\n- ¼ cup cream or milk, optional\n- A few basil leaves, optional\n\n### Method\n1. Heat the oil in a pot over medium heat. Add the onion and cook for 4–5 minutes until softened. Stir in the garlic and cook for 30 seconds.\n2. Add the tomatoes, broth, sugar if using, and a pinch of salt and pepper. Bring to a simmer and cook for 10 minutes.\n3. Blend until smooth with an immersion blender, or carefully blend in batches.\n4. Stir in the cream, if you like, and adjust the seasoning. Serve with basil and toast or grilled cheese.'